# Laboratorijska vježba 6: Konvolucione neuronske mreže (CNN)

---
## Zadatak 1 – Korištenje CNN za klasifikaciju (CIFAR-10)

### a) Učitavanje CIFAR-10 skupa podataka i ispis oblika podataka

In [ ]:
import numpy as np
from tensorflow.keras.datasets import cifar10

# Real CIFAR-10 images and labels; first run downloads the dataset.
# The original coursework copy used random pixels/labels as a shape placeholder.
(trainX, trainy), (testX, testy) = cifar10.load_data()

print('Oblik trening podataka (slike):', trainX.shape)
print('Oblik trening labela:', trainy.shape)
print('Oblik test podataka (slike):', testX.shape)
print('Oblik test labela:', testy.shape)


### b) Vizualizacija prvih 9 slika iz CIFAR-10 skupa

In [ ]:
import matplotlib.pyplot as plt

# Nazivi klasa
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(trainX[i])
    ax.set_title(f'Klasa: {class_names[trainy[i][0]]}', fontsize=10)
    ax.axis('off')

plt.suptitle('Prvih 9 slika iz CIFAR-10 skupa podataka', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""Zaključak:
- Slike su dimenzija 32x32 piksela u RGB formatu.
- Rezolucija je vrlo niska u poređenju s modernim fotografijama (npr. 12MP).
- Niska rezolucija otežava klasifikaciju jer se gube sitni detalji objekta,
  ali ubrzava treniranje i smanjuje memorijsku potrošnju.
- Modeli mašinskog učenja su osjetljivi na rezoluciju – bolji rezultati
  se generalno postižu sa višom rezolucijom, ali cijena računanja raste.""")

### c) Priprema (predprocesiranje) podataka

In [ ]:
from keras.utils import to_categorical

# One-hot enkodiranje labela
trainY = to_categorical(trainy)
testY  = to_categorical(testy)

# Normalizacija slika: float32, raspon [0, 1]
trainX = trainX.astype('float32') / 255.0
testX  = testX.astype('float32')  / 255.0

print('Oblik trainY (one-hot):', trainY.shape)
print('Oblik testY  (one-hot):', testY.shape)
print('Min/Max vrijednost piksela – train:', trainX.min(), '/', trainX.max())

# Reserve validation data from training; keep the official test set held out.
from sklearn.model_selection import train_test_split
trainX, valX, trainY, valY = train_test_split(
    trainX, trainY, test_size=0.1, random_state=42, stratify=trainY.argmax(axis=1)
)


### d) & e) Definisanje baseline modela

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.optimizers import SGD

def define_model():
    model = Sequential()

    # Blok 1
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    # Blok 2
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    # Klasifikacijska glava
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    # Kompajliranje
    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_baseline = define_model()
model_baseline.summary()

### f) Treniranje i evaluacija baseline modela

In [ ]:
history_baseline = model_baseline.fit(
    trainX, trainY,
    epochs=15,
    batch_size=64,
    validation_data=(valX, valY),
    verbose=1
)

_, acc_baseline = model_baseline.evaluate(testX, testY, verbose=0)
print(f'\nTačnost baseline modela na testnom skupu: {acc_baseline * 100:.2f}%')
print('Ovo je solidan početni rezultat, ali postoji prostor za poboljšanje.')

### g) Plot tačnosti i gubitka – baseline model

In [ ]:
def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Tačnost
    ax1.plot(history.history['accuracy'],     label='Trening tačnost',    color='royalblue')
    ax1.plot(history.history['val_accuracy'], label='Validacijska tačnost', color='darkorange')
    ax1.set_title(f'Tačnost {title}', fontsize=12)
    ax1.set_xlabel('Epoha')
    ax1.set_ylabel('Tačnost')
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)

    # Gubitak
    ax2.plot(history.history['loss'],     label='Trening gubitak',    color='royalblue')
    ax2.plot(history.history['val_loss'], label='Validacijski gubitak', color='darkorange')
    ax2.set_title(f'Gubitak {title}', fontsize=12)
    ax2.set_xlabel('Epoha')
    ax2.set_ylabel('Gubitak')
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

plot_history(history_baseline, '– Baseline model')

print("""Zaključak (baseline):
- Trening tačnost raste konzistentno kroz epohe.
- Validacijska tačnost je niža i može stagnirati – znak mogućeg overfittinga.
- Gubitak na treningu opada, dok validacijski gubitak može porasti – potvrda overfittinga.""")

### h) Dodavanje Dropout sloja (p=0.5) – poboljšani model

In [ ]:
from keras.layers import Dropout

def define_model_dropout():
    model = Sequential()

    # Blok 1
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    # Blok 2
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    # Klasifikacijska glava
    model.add(Flatten())
    model.add(Dropout(0.5))          # Dropout prije prvog Dense sloja
    model.add(Dense(128, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_dropout = define_model_dropout()
model_dropout.summary()

In [ ]:
history_dropout = model_dropout.fit(
    trainX, trainY,
    epochs=15,
    batch_size=64,
    validation_data=(valX, valY),
    verbose=1
)

_, acc_dropout = model_dropout.evaluate(testX, testY, verbose=0)
print(f'\nTačnost modela s Dropout(0.5): {acc_dropout * 100:.2f}%')

plot_history(history_dropout, '– Model s Dropout(0.5)')

print("""Zaključak (Dropout 0.5):
- Dropout pomaže u smanjenju overfittinga jer nasumično isključuje neurone tokom treniranja.
- Razlika između trening i validacijske tačnosti trebala bi biti manja nego kod baseline modela.
- Eventualno može biti nešto niža trening tačnost, ali bolja generalizacija.""")

### i) Finalni model – tri konvoluciona bloka + Dropout(0.2)

In [ ]:
def define_final_model():
    model = Sequential()

    # Blok 1 – 32 filtera
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    # Blok 2 – 64 filtera
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    # Blok 3 – 128 filtera
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    # Klasifikacijska glava
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.2))          # Dropout nakon prvog Dense sloja
    model.add(Dense(10, activation='softmax'))

    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_final = define_final_model()
model_final.summary()

In [ ]:
history_final = model_final.fit(
    trainX, trainY,
    epochs=30,
    batch_size=64,
    validation_data=(valX, valY),
    verbose=1
)

_, acc_final = model_final.evaluate(testX, testY, verbose=0)
print(f'\nTačnost finalnog modela na testnom skupu: {acc_final * 100:.2f}%')

plot_history(history_final, '– Finalni model')

print(f"""
Zaključak (finalni model):
- Postignuta tačnost: {acc_final * 100:.2f}%
- Treći konvolucioni blok (128 filtera) povećava kapacitet modela za učenje složenijih značajki.
- Dropout(0.2) nakon svakog pooling sloja i Dense sloja smanjuje overfitting.
- Uz 30 epoha i regularizaciju, model pokazuje bolju generalizaciju od baseline modela.
- Daljnje poboljšanje je moguće s: weight decay, batch normalization, data augmentation,
  ili primjenom transfer learninga (npr. VGG16, ResNet).""")

### Usporedba svih modela

In [ ]:
print('=' * 45)
print(f'{"Model":<25} {"Tačnost":>10}')
print('=' * 45)
print(f'{"Baseline":.<25} {acc_baseline*100:>9.2f}%')
print(f'{"Dropout(0.5)":.<25} {acc_dropout*100:>9.2f}%')
print(f'{"Finalni model":.<25} {acc_final*100:>9.2f}%')
print('=' * 45)